In [1]:
import gymnasium as gym
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy

In [2]:
# 创建CartPole-v1环境和PPO模型
env = gym.make("CartPole-v1")

model = PPO(MlpPolicy, env, verbose=0)

In [3]:
from stable_baselines3.common.base_class import BaseAlgorithm


def evaluate(
    model: BaseAlgorithm,
    num_episodes: int = 100,
    deterministic: bool = True,
) -> float:
    """
    Evaluate an RL agent for `num_episodes`.

    :param model: the RL Agent
    :param env: the gym Environment
    :param num_episodes: number of episodes to evaluate it
    :param deterministic: Whether to use deterministic or stochastic actions
    :return: Mean reward for the last `num_episodes`
    """
    # This function will only work for a single environment
    vec_env = model.get_env()
    obs = vec_env.reset()
    all_episode_rewards = []
    for _ in range(num_episodes):
        episode_rewards = []
        done = False
        # Note: SB3 VecEnv resets automatically:
        # https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html#vecenv-api-vs-gym-api
        # obs = vec_env.reset()
        while not done:
            # _states are only useful when using LSTM policies
            # `deterministic` is to use deterministic actions
            action, _states = model.predict(obs, deterministic=deterministic)
            # here, action, rewards and dones are arrays
            # because we are using vectorized env
            obs, reward, done, _info = vec_env.step(action)
            episode_rewards.append(reward)

        all_episode_rewards.append(sum(episode_rewards))

    mean_episode_reward = np.mean(all_episode_rewards)
    print(f"Mean reward: {mean_episode_reward:.2f} - Num episodes: {num_episodes}")

    return mean_episode_reward

In [4]:
# 训练前评估（随机策略）
mean_reward_before_train = evaluate(model, num_episodes=100, deterministic=True)

Mean reward: 9.40 - Num episodes: 100


在训练前，使用随机策略在100个回合中的平均奖励为9.4，这个分数相对较低，表明智能体还没有学会如何有效平衡杆子

In [5]:
# 使用Stable-Baselines3内置评估函数

from stable_baselines3.common.evaluation import evaluate_policy

mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=100, warn=False)

print(f"mean_reward: {mean_reward:.2f} +/- {std_reward:.2f}")

mean_reward: 9.24 +/- 0.74


使用Stable-Baselines3内置的评估函数，平均奖励9.24，标准差0.74，结果与自定义评估函数基本一致

In [6]:
# 训练智能体，进行100000个时间步的训练
model.learn(total_timesteps=100000)

In [7]:
# 训练后评估
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=100)

print(f"mean_reward:{mean_reward:.2f} +/- {std_reward:.2f}")

/opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


mean_reward:500.00 +/- 0.00


训练后平均奖励大幅提升到500，标准差0，说明每次回合奖励都是500分，智能体已经学会了如何更好地平衡杆子，性能显著提升

In [8]:
# Set up fake display; otherwise rendering will fail
import os
os.system("Xvfb :1 -screen 0 1024x768x24 &")
os.environ['DISPLAY'] = ':1'

sh: Xvfb: command not found


In [9]:
import base64
from pathlib import Path

from IPython import display as ipythondisplay


def show_videos(video_path="", prefix=""):
    """
    Taken from https://github.com/eleurent/highway-env

    :param video_path: (str) Path to the folder containing videos
    :param prefix: (str) Filter the video, showing only the only starting with this prefix
    """
    html = []
    for mp4 in Path(video_path).glob("{}*.mp4".format(prefix)):
        video_b64 = base64.b64encode(mp4.read_bytes())
        html.append(
            """<video alt="{}" autoplay 
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{}" type="video/mp4" />
                </video>""".format(
                mp4, video_b64.decode("ascii")
            )
        )
    ipythondisplay.display(ipythondisplay.HTML(data="<br>".join(html)))

In [10]:
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv


def record_video(env_id, model, video_length=500, prefix="", video_folder="videos/"):
    """
    :param env_id: (str)
    :param model: (RL model)
    :param video_length: (int)
    :param prefix: (str)
    :param video_folder: (str)
    """
    eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])
    # Start the video at step=0 and record 500 steps
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder=video_folder,
        record_video_trigger=lambda step: step == 0,
        video_length=video_length,
        name_prefix=prefix,
    )

    obs = eval_env.reset()
    for _ in range(video_length):
        action, _ = model.predict(obs)
        obs, _, _, _ = eval_env.step(action)

    # Close the video recorder
    eval_env.close()

In [11]:
record_video("CartPole-v1", model, video_length=500, prefix="ppo-cartpole")

objc[49321]: Class SDL_RumbleMotor is implemented in both /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x11bef8d40) and /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x12e2849c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[49321]: Class SDL_RumbleContext is implemented in both /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x11bef8d90) and /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x12e284a18). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[49321]: Class SDLApplication is implemented in both /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x11bef8890) and /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/pyg

Saving video to /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/lab/videos/ppo-cartpole-step-0-to-step-500.mp4
MoviePy - Building video /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/lab/videos/ppo-cartpole-step-0-to-step-500.mp4.
MoviePy - Writing video /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/lab/videos/ppo-cartpole-step-0-to-step-500.mp4



MoviePy - Done !
MoviePy - video ready /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/lab/videos/ppo-cartpole-step-0-to-step-500.mp4


In [12]:
show_videos("videos", prefix="ppo")

还有更简洁的写法，如下所示，这行代码完成了以下工作：
- 自动创建环境：基于 "CartPole-v1" 创建强化学习环境
- 初始化模型：使用MLP策略网络初始化PPO算法
- 开始训练：进行100000个时间步的训练
- 返回模型：将训练好的智能体赋值给变量供后续使用

In [13]:
model = PPO('MlpPolicy', "CartPole-v1", verbose=1).learn(100000) # verbose=1: 设置训练过程中的输出详细程度为1级（显示训练进度）

Using cpu device
Creating environment from the given name 'CartPole-v1'
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 23.8     |
|    ep_rew_mean     | 23.8     |
| time/              |          |
|    fps             | 10089    |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 27.2         |
|    ep_rew_mean          | 27.2         |
| time/                   |              |
|    fps                  | 6398         |
|    iterations           | 2            |
|    time_elapsed         | 0            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0075779893 |
|    clip_fraction        | 0.089     

In [14]:
record_video("CartPole-v1", model, video_length=500, prefix="ppo-cartpole-oneline")

Saving video to /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/lab/videos/ppo-cartpole-oneline-step-0-to-step-500.mp4
MoviePy - Building video /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/lab/videos/ppo-cartpole-oneline-step-0-to-step-500.mp4.
MoviePy - Writing video /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/lab/videos/ppo-cartpole-oneline-step-0-to-step-500.mp4



MoviePy - Done !
MoviePy - video ready /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/lab/videos/ppo-cartpole-oneline-step-0-to-step-500.mp4


In [16]:
show_videos("videos", prefix="ppo-cartpole-oneline")